In [1]:
import random
import torch
import os
import re

import pandas as pd
import polars as pl
import numpy as np

import sys
sys.path.append('../')
import compcor.corpus_metrics as corpus_metrics
from compcor.utils import Corpus
from compcor.text_tokenizer_embedder import STTokenizerEmbedder

from collections import defaultdict

/home/isabel/anaconda3/envs/localSyntheticData/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

In [2]:


# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)



In [28]:
# ------------------ Metric Setup (experiment_config.py) ------------------
comp_metrics = [
	corpus_metrics.chi_square_distance,
	corpus_metrics.zipf_distance,
	corpus_metrics.classifier_distance,
	corpus_metrics.IRPR_distance,
	corpus_metrics.fid_distance,
	corpus_metrics.pr_distance,
	corpus_metrics.dc_distance,
	corpus_metrics.mauve_distance,
	corpus_metrics.traditional_biber_distance,
	corpus_metrics.zero_wasserstein_distance
]

metrics_names = [((str(dist).split()[1]).split('_')[0]).upper() for dist in comp_metrics]

# Helper function for getting metric-dependent data.
def get_metric_dependant_data(metric, corpus: Corpus):
	if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
		c = STTokenizerEmbedder().tokenize_sentences(corpus)
	elif metric == corpus_metrics.zero_biber_distance:
		c = corpus
	else:
		c = STTokenizerEmbedder().embed_sentences(corpus)
	return c

# Helper function for getting all metric data.
def get_data_for_compcor_metrics(corpus):
	tokens = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").tokenize_sentences(corpus)
	embeddings = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").embed_sentences(corpus)
	return tokens, embeddings

def get_distances_from_compare_corpora(setA, setB):    
    tokensA, embeddingsA = get_data_for_compcor_metrics(setA)
    tokensB, embeddingsB = get_data_for_compcor_metrics(setB)
    distances = {}
    for metric_name, metric in zip(metrics_names, comp_metrics):
        if metric in (corpus_metrics.zipf_distance, corpus_metrics.chi_square_distance):
            tempA, tempB = tokensA, tokensB
        elif metric in (corpus_metrics.traditional_biber_distance, corpus_metrics.zero_wasserstein_distance):
            tempA, tempB = setA, setB
        else:
            tempA, tempB = embeddingsA, embeddingsB
        distances[metric_name] = metric(corpus1=tempA, corpus2=tempB)

    return distances


In [4]:
# subject the real data to the same processing as the generated data
def clean_note(text):
    text = re.sub(r'^[\s"]+|[\s"]+$', '', text)   # strip edge quotes/spaces
    text = re.sub(r'\*', '', text)                 # remove asterisks
    text = re.sub(r'\s+', ' ', text)              # normalize whitespace
    return text.strip()

real_datasets = {}
for folder in os.listdir('./getText/datasetsPrep/'):
    if os.path.isdir(f'./getText/datasetsPrep/{folder}'):
        for dataset in os.listdir(f'./getText/datasetsPrep/{folder}'):
            temp_df = pd.read_csv(f'./getText/datasetsPrep/{folder}/{dataset}', index_col=None)
            temp_df = temp_df.dropna(subset='text')
            temp_df['text'] = [clean_note(text) for text in temp_df['text'].tolist()]
            real_datasets[dataset.replace('.csv', '')] = temp_df

In [5]:
generated_datasets = {}
for dataset in os.listdir(f'./dataGeneration/processedData/'):
    temp_df = pd.read_csv(f'./dataGeneration/processedData/{dataset}', index_col=None)
    generated_datasets[dataset.replace('.csv', '')] = {
        'LDA': temp_df[temp_df['topic_model'] == 'LDA'],# ['report'].dropna().tolist(),
        'MATAVE': temp_df[temp_df['topic_model'] == 'MATAVE']
    }

In [6]:
assert generated_datasets.keys() == real_datasets.keys(), "Keys for both real and generated datasets must be the same."

dataset_metrics_averaged = {}
for dataset, topic_model_dict in generated_datasets.items():
    dataset_metrics_averaged[dataset] = {'LDA': {}, 'MATAVE': {}}
    temp_models = {'LDA': defaultdict(list), 'MATAVE': defaultdict(list)}
    for i in range(3):
        real_sample = (real_datasets[dataset].sample(frac=1, random_state = i)['text'].tolist())[:100]
        for model in temp_models:
            model_sample = (topic_model_dict[model].sample(frac=1, random_state = i)['report'].dropna().tolist())[:100]
            temp_metrics = get_distances_from_compare_corpora(real_sample, model_sample)

            for metric, value in temp_metrics.items():
                temp_models[model][metric].append(value)
                                                  
    for model, metrics_dict in temp_models.items():
            for metric, values in metrics_dict.items():
                dataset_metrics_averaged[dataset][model][metric] = sum(values) / len(values)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

The `tokenize` method is deprecated, please use `preprocess` instead.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Num real: 100 Num fake: 100
Num real: 100 Num fake: 100


WARNING clustering 200 points to 10 centroids: please provide at least 390 training points
[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_02_perfect_aspect', 'f_08_third_person_pronouns', 'f_09_pronoun_it', 'f_12_proverb_do', 'f_18_by_passives', 'f_21_that_verb_comp', 'f_22_that_adj_comp', 'f_23_wh_clause', 'f_26_past_participle', 'f_30_that_obj', 'f_32_wh_obj', 'f_33_pied_piping', 'f_34_sentence_relatives', 'f_35_because', 'f_36_though', 'f_37_if', 'f_38_other_adv_sub', 'f_45_conjuncts', 'f_47_hedges', 'f_48_amplifiers', 'f_50_discourse_particles', 'f_55_verb_public', 'f_58_verb_seem', 'f_60_that_deletion', 'f_61_stranded_preposition', 'f_66_neg_synthetic', 'f_67_neg_analytic']


In [29]:
for dataset, _ in generated_datasets.items():
    dataset_metrics_averaged[dataset]['realToReal'] = {}
    temp_models = {'realToReal': defaultdict(list)}
    for i in range(3):
        real_sample = real_datasets[dataset].sample(frac=1, random_state = i)['text'].tolist()
        assert len(real_sample) >= 200, "real samples must be over 200 samples long to prevent bias"
        real_start = real_sample[:100]
        real_end = real_sample[-100:]


        temp_metrics = get_distances_from_compare_corpora(real_start, real_end)

        for temp_metric, value in temp_metrics.items():
            temp_models['realToReal'][temp_metric].append(value)
                                                  
    for model, metrics_dict in temp_models.items():
            for temp_metric, values in metrics_dict.items():
                dataset_metrics_averaged[dataset][model][temp_metric] = sum(values) / len(values)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Num real: 100 Num fake: 100
Num real: 100 Num fake: 100


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Num real: 100 Num fake: 100
Num real: 100 Num fake: 100


WARNING clustering 200 points to 10 centroids: please provide at least 390 training points
[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_01_past_tense', 'f_02_perfect_aspect', 'f_08_third_person_pronouns', 'f_10_demonstrative_pronoun', 'f_12_proverb_do', 'f_18_by_passives', 'f_21_that_verb_comp', 'f_22_that_adj_comp', 'f_25_present_participle', 'f_26_past_participle', 'f_30_that_obj', 'f_32_wh_obj', 'f_33_pied_piping', 'f_34_sentence_relatives', 'f_35_because', 'f_36_though', 'f_37_if', 'f_38_other_adv_sub', 'f_45_conjuncts', 'f_47_hedges', 'f_48_amplifiers', 'f_50_discourse_particles', 'f_55_verb_public', 'f_58_verb_seem', 'f_60_that_deletion', 'f_61_stranded_preposition', 'f_66_neg_synthetic', 'f_67_neg_analytic']


In [32]:
for dataset, _ in generated_datasets.items():
    dataset_metrics_averaged[dataset]['realToReal2'] = {}
    temp_models = {'realToReal2': defaultdict(list)}
    for i in range(3):
        real_sample = real_datasets[dataset].sample(frac=1, random_state = i)['text'].tolist()
        assert len(real_sample) >= 200, "real samples must be over 200 samples long to prevent bias"
        real_start = real_sample[:100]
        real_end = real_sample[100:200]


        temp_metrics = get_distances_from_compare_corpora(real_start, real_end)

        for temp_metric, value in temp_metrics.items():
            temp_models['realToReal2'][temp_metric].append(value)
                                                  
    for model, metrics_dict in temp_models.items():
            for temp_metric, values in metrics_dict.items():
                dataset_metrics_averaged[dataset][model][temp_metric] = sum(values) / len(values)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Num real: 100 Num fake: 100
Num real: 100 Num fake: 100


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Num real: 100 Num fake: 100
Num real: 100 Num fake: 100


WARNING clustering 200 points to 10 centroids: please provide at least 390 training points
[INFO] Using TTR for f_43_type_token
[INFO] All features normalized per 1000 tokens except: f_43_type_token and f_44_mean_word_length
INFO:pybiber.biber_analyzer:Zero-variance features retained (neutral scaling) in projection: ['f_02_perfect_aspect', 'f_08_third_person_pronouns', 'f_09_pronoun_it', 'f_10_demonstrative_pronoun', 'f_12_proverb_do', 'f_18_by_passives', 'f_21_that_verb_comp', 'f_22_that_adj_comp', 'f_23_wh_clause', 'f_25_present_participle', 'f_26_past_participle', 'f_30_that_obj', 'f_32_wh_obj', 'f_33_pied_piping', 'f_34_sentence_relatives', 'f_35_because', 'f_36_though', 'f_37_if', 'f_38_other_adv_sub', 'f_45_conjuncts', 'f_46_downtoners', 'f_47_hedges', 'f_48_amplifiers', 'f_49_emphatics', 'f_50_discourse_particles', 'f_55_verb_public', 'f_58_verb_seem', 'f_60_that_deletion', 'f_61_stranded_preposition', 'f_66_neg_synthetic', 'f_67_neg_analytic']


In [33]:
rows = []
for dataset, model_dict in dataset_metrics_averaged.items():
    for model, metrics in model_dict.items():
        temp_dict = {
            'dataset': dataset,
            'model': model,

        }
        for metric, value in metrics.items():
            temp_dict[metric] = value
        rows.append(temp_dict)

df = pd.DataFrame(rows)
df.to_csv("./ldaMataveMetrics.csv", index=False)

In [34]:
df

,dataset,model,CHI,ZIPF,CLASSIFIER,IRPR,FID,PR,DC,MAUVE,TRADITIONAL,ZERO
0,yahoo,LDA,9.436303e-01,0.122554,0.881197,0.412986,1.012399,0.349915,0.103568,0.713425,0.374836,0.270910
1,yahoo,MATAVE,9.916591e-01,0.222275,0.956306,0.416155,1.028472,0.376042,0.096035,0.772034,0.562229,0.295235
2,yahoo,realToReal,1.000000e+00,0.005036,0.472527,0.403372,0.880048,0.010016,0.029556,0.039246,0.148319,0.100081
3,yahoo,realToReal2,1.000000e+00,0.007488,0.497258,0.402451,0.863901,0.015110,0.013775,0.022787,0.179956,0.101998
4,banking77,LDA,2.776298e-13,0.131400,0.907309,0.310814,0.615170,0.382805,0.308852,0.703925,0.365555,0.147635
5,banking77,MATAVE,6.055634e-03,0.176033,0.975203,0.337082,0.716313,0.469054,0.637870,0.882104,0.445697,0.220855
6,banking77,realToReal,1.000000e+00,0.018521,0.556985,0.258443,0.365246,0.038515,0.007754,0.028652,0.154659,0.084253
7,banking77,realToReal2,1.000000e+00,0.015546,0.534355,0.259503,0.369778,0.036894,0.015349,0.026769,0.171889,0.081735
8,medicalAbstracts,LDA,0.000000e+00,0.014930,0.763263,0.354773,0.708207,0.159788,0.034525,0.285763,0.501198,0.532371
9,medicalAbstracts,MATAVE,0.000000e+00,0.004101,0.931579,0.360913,0.972905,0.501161,0.197862,0.955450,0.533052,0.494359
